# RedBus Data Analysis

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
#Load path where my dataset is
path = "/content/drive/MyDrive/RedBusAnalysis"

In [14]:
#Importing pandas and numpy
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
import numpy as np

In [15]:
#Loading data set
train_data = pd.read_csv(path + "/train.csv")
transaction_data = pd.read_csv(path + "/transactions.csv")
test_data = pd.read_csv(path + "/test_8gqdJqH.csv")

# Feature Extraction

In [20]:
#Filtering for prediction 15 days before journey
transaction_15 = transaction_data[transaction_data["dbd"] == 15]

In [21]:
#Creating unique route key to match later with test dataset
transaction_15["route_key"] = transaction_15["doj"] + "_" + transaction_15["srcid"].astype(str) + "_" + transaction_15["destid"].astype(str)

/tmp/ipython-input-21-1703587420.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  transaction_15["route_key"] = transaction_15["doj"] + "_" + transaction_15["srcid"].astype(str) + "_" + transaction_15["destid"].astype(str)


In [22]:
transaction_15 = transaction_15.dropna()

In [23]:
#Selecting Relevant feature
features = transaction_15[["route_key", "cumsum_seatcount", "cumsum_searchcount", "srcid_region", "destid_region", "srcid_tier", "destid_tier"]]

In [24]:
#Merge with train labels
train_data["route_key"] = train_data["doj"].astype(str) + "_" + train_data["srcid"].astype(str) + "_" + train_data["destid"].astype(str)
#Drop if existing feature in train data to avoid collision during merge
cols_to_drop = ["cumsum_seatcount", "cumsum_searchcount",
                "srcid_region", "destid_region", "srcid_tier", "destid_tier"]

train_data = train_data.drop(columns=[col for col in cols_to_drop if col in train_data.columns])

train_data = train_data.merge(features, on="route_key", how="left")
train_data.dropna(inplace=True)

In [25]:
#Mergin with test set
duplicate_cols = [
    "cumsum_seatcount", "cumsum_searchcount",
    "srcid_region", "destid_region",
    "srcid_tier", "destid_tier"
]

# Drop them from test_data if they exist
test_data = test_data.drop(columns=[col for col in duplicate_cols if col in test_data.columns])
test_data = test_data.merge(features, on="route_key", how="left")
test_data['cumsum_seatcount'] = test_data['cumsum_seatcount'].fillna(0)
test_data['cumsum_searchcount'] = test_data['cumsum_searchcount'].fillna(0)
for col in ["srcid_region", "destid_region", "srcid_tier", "destid_tier"]:
    test_data[col] = test_data[col].fillna("Unknown")

In [26]:
#Adding seat search ratio -- features
train_data["seat_to_search_ratio"] = train_data["cumsum_seatcount"] / (train_data["cumsum_searchcount"] + 1)
test_data["seat_to_search_ratio"] = test_data["cumsum_seatcount"] / (test_data["cumsum_searchcount"] + 1)

In [27]:
#Log Transform (Stabilize scale) -- features
for df in [train_data, test_data]:
    df["log_seatcount"] = np.log1p(df["cumsum_seatcount"])  # log(1 + x)
    df["log_searchcount"] = np.log1p(df["cumsum_searchcount"])

In [28]:
#Extract day of week and Month from object
train_data["doj"] = pd.to_datetime(train_data["doj"])
test_data["doj"] = pd.to_datetime(test_data["doj"])

# Extract calendar features
for df in [train_data, test_data]:
    df["doj_dayofweek"] = df["doj"].dt.dayofweek  # 0=Monday
    df["doj_month"] = df["doj"].dt.month

In [29]:
#String based interaction (categorical model can handle it)
# Label encode it (or use frequency encoding)
train_data["route_pair"] = train_data["srcid"].astype(str) + "_" + train_data["destid"].astype(str)
test_data["route_pair"] = test_data["srcid"].astype(str) + "_" + test_data["destid"].astype(str)

# Label encode it (or use frequency encoding)
from sklearn.preprocessing import LabelEncoder
le_route = LabelEncoder()
combined_routes = pd.concat([train_data["route_pair"], test_data["route_pair"]])
le_route.fit(combined_routes)

train_data["route_pair_enc"] = le_route.transform(train_data["route_pair"])
test_data["route_pair_enc"] = le_route.transform(test_data["route_pair"])

In [30]:
#Target Encoding (Mean Encoding)
def target_encode(train_df, test_df, cat_col, target_col="final_seatcount"):
    target_map = train_df.groupby(cat_col)[target_col].mean()
    train_df[f"{cat_col}_target"] = train_df[cat_col].map(target_map)
    test_df[f"{cat_col}_target"] = test_df[cat_col].map(target_map)  # Use same map (no leakage)
    test_df[f"{cat_col}_target"] = test_df[f"{cat_col}_target"].fillna(target_map.mean())

In [31]:
categorical_features = ["srcid_region", "destid_region", "srcid_tier", "destid_tier", "route_pair"]
for col in categorical_features:
    target_encode(train_data, test_data, col)

In [32]:
#Frequency Encoding
def frequency_encode(train_df, test_df, col):
    freq_map = train_df[col].value_counts(normalize=False)
    train_df[f"{col}_freq"] = train_df[col].map(freq_map)
    test_df[f"{col}_freq"] = test_df[col].map(freq_map)
    test_df[f"{col}_freq"] = test_df[f"{col}_freq"].fillna(0)  # Handle unseen

In [33]:
for col in categorical_features:
    frequency_encode(train_data, test_data, col)

In [35]:
features = [
    "cumsum_seatcount", "cumsum_searchcount", "seat_to_search_ratio",
    "log_seatcount", "log_searchcount",
    "doj_dayofweek", "doj_month",
    "srcid_region_target", "destid_region_target",
    "srcid_tier_target", "destid_tier_target",
    "route_pair_target"
]
target = "final_seatcount"

In [43]:
X = train_data[features]
y = train_data[target]
X_test = test_data[features]

# Model training

In [58]:
# Split data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [59]:
from sklearn.ensemble import RandomForestRegressor

In [70]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [71]:
y_pred = model.predict(X_val)

In [72]:
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print("Random Forest RMSE:", rmse)

Random Forest RMSE: 471.00623674695703


In [73]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

LinearRegression()

In [74]:
y_pred_linear = linear_model.predict(X_val)

In [75]:
rmse = np.sqrt(mean_squared_error(y_val, y_pred_linear))
print("Random Forest RMSE:", rmse)

Random Forest RMSE: 650.0106378706437


In [76]:
random_pred = model.predict(X_test)

In [77]:
linear_pred = linear_model.predict(X_test)

In [83]:
final_pred = 0.5 * random_pred + 0.3 * linear_pred

In [84]:
final_pred

array([2884.13733067, 1055.8069966 ,  578.56525276, ..., 1547.10220734,
        678.92414504, 1652.82101812])

In [85]:
submission = test_data[["route_key"]].copy()
submission["final_seatcount"] = final_pred.round().astype(int)
submission.to_csv("submission_file.csv", index=False)

In [86]:
#Download submission file
from google.colab import files
files.download("submission_file.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [87]:
submission

,route_key,final_seatcount
0,2025-02-11_46_45,2884
1,2025-01-20_17_23,1056
2,2025-01-08_02_14,579
3,2025-01-08_08_47,556
4,2025-01-08_09_46,2393
...,...,...
5895,2025-01-23_46_48,3523
5896,2025-02-21_46_09,2448
5897,2025-01-17_32_19,1547
5898,2025-01-24_45_03,679
